# Data Preprocessing
## CVD Fairness Dissertation — NB0

**Purpose:** Load raw data, recode gender, convert age, remove clinically implausible records, and save the cleaned dataset.

**Input:** `data/raw/cardio_train.csv`  
**Output:** `data/processed/cardio_baseline_clean.csv`

**Does NOT contain:** EDA, distributions, correlations — those are in `02_eda.ipynb`

### Dataset Description

There are 3 types of input features:

*Objective*: factual information; *Examination*: results of medical examination; *Subjective*: information given by the patient.

| Feature | Type | Notes |
|---|---|---|
| age | Objective | int (days) |
| height | Objective | int (cm) |
| weight | Objective | float (kg) |
| gender | Objective | categorical code |
| ap_hi | Examination | Systolic blood pressure |
| ap_lo | Examination | Diastolic blood pressure |
| cholesterol | Examination | 1: normal, 2: above normal, 3: well above normal |
| gluc | Examination | 1: normal, 2: above normal, 3: well above normal |
| smoke | Subjective | binary |
| alco | Subjective | binary |
| active | Subjective | binary |
| cardio | Target | binary — presence/absence of CVD diagnosis |

**Limitation:** The target variable records diagnostic outcome rather than confirmed biological disease presence. Given well-documented systematic underdiagnosis of CVD in women, a negative label cannot be assumed to reliably confirm absence of disease in female patients.

## 1. Imports and Paths

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

RAW_DATA   = Path("../raw/cardio_train.csv")
OUTPUT_DIR = Path("../../outputs/preprocessing")
CLEAN_OUT  = Path("../data/processed/cardio_baseline_clean.csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Raw Data and Recode Gender

Gender is recoded from the original encoding (1=female, 2=male) to a clearer binary (0=female, 1=male) used consistently throughout the project.

In [4]:
data = pd.read_csv(RAW_DATA, sep=",")
data["gender"] = data["gender"].replace({1: 0, 2: 1})  # 0=female, 1=male

assert set(data["gender"].unique()) == {0, 1}, "Unexpected gender values after recode"
assert data.isnull().sum().sum() == 0, "Null values found in raw data"

print(f"Raw dataset shape : {data.shape}")
print(f"Gender values     : {sorted(data['gender'].unique())}  (0=female, 1=male)")
print(f"Female            : {(data['gender']==0).sum():,}  ({(data['gender']==0).mean()*100:.1f}%)")
print(f"Male              : {(data['gender']==1).sum():,}  ({(data['gender']==1).mean()*100:.1f}%)")

Raw dataset shape : (70000, 13)
Gender values     : [np.int64(0), np.int64(1)]  (0=female, 1=male)
Female            : 45,530  (65.0%)
Male              : 24,470  (35.0%)


## 3. Age Conversion and BMI

Age is converted from days to years. BMI is computed temporarily to help identify implausible height/weight combinations but is not retained in the final dataset.

In [ ]:
data["age_years"] = data["age"] / 365.25
data["BMI"] = data["weight"] / (data["height"] / 100) ** 2

print("Age (years) summary:")
print(data["age_years"].describe().round(1))
print(f"\nBMI > 70 count : {(data['BMI'] > 70).sum()}")

## 4. Identify Implausible Values

Before applying filters, the extent of implausible values is checked to justify the exclusion thresholds used below.

In [ ]:
print("Implausible blood pressure values in raw data:")
print(f"  Systolic > 250       : {(data['ap_hi'] > 250).sum()}  (range: {data[data['ap_hi']>250]['ap_hi'].min()}–{data[data['ap_hi']>250]['ap_hi'].max()})")
print(f"  Diastolic > 150      : {(data['ap_lo'] > 150).sum()}  (range: {data[data['ap_lo']>150]['ap_lo'].min()}–{data[data['ap_lo']>150]['ap_lo'].max()})")
print(f"  Systolic < 70        : {(data['ap_hi'] < 70).sum()}")
print(f"  Diastolic < 40       : {(data['ap_lo'] < 40).sum()}")
print(f"  Diastolic > Systolic : {(data['ap_lo'] > data['ap_hi']).sum()}")

## 5. Apply Filters

Records with clinically implausible values are removed. Each filter is logged showing how many records were kept and removed at each step.

In [ ]:
def apply_filter(df, mask, name):
    before = len(df)
    df = df[mask].copy()
    after = len(df)
    print(f"{name:35s} | kept {after:6d}/{before:6d} | removed {before - after:6d}")
    return df

df = data.copy()

df = apply_filter(df, (df["age_years"] >= 30) & (df["age_years"] <= 65),  "Age 30–65 years")
df = apply_filter(df, (df["height"] >= 140)   & (df["height"] <= 220),    "Height 140–220 cm")
df = apply_filter(df, (df["weight"] >= 45)    & (df["weight"] <= 200),    "Weight 45–200 kg")
df = apply_filter(df, (df["ap_hi"] >= 70)     & (df["ap_hi"] <= 200),     "Systolic (ap_hi) 70–200")
df = apply_filter(df, (df["ap_lo"] >= 40)     & (df["ap_lo"] <= 150),     "Diastolic (ap_lo) 40–150")
df = apply_filter(df, df["ap_lo"] <= df["ap_hi"],                          "Diastolic <= Systolic")
df = apply_filter(df, df["cholesterol"].isin([1, 2, 3]),                   "Cholesterol in {1,2,3}")
df = apply_filter(df, df["gluc"].isin([1, 2, 3]),                          "Glucose in {1,2,3}")
for col in ["smoke", "alco", "active", "cardio"]:
    df = apply_filter(df, df[col].isin([0, 1]), f"{col} in {{0,1}}")

df = df.drop(columns=["id", "age", "BMI"])

print(f"\nFinal cleaned shape: {df.shape}")